# ISAN 3392 - Model B: placement-readiness risk, and composing A with B
**Alibaba ASI trace (OSDI '26) - D2_PLACEMENT.** Plan RQ2 and RQ3: a *calibrated* probability that a pod fails to become ready at admission,
three model families (B1 logistic, B2 random forest, B3 LightGBM), leave-one-cluster-out, calibration diagnostics, and the A+B policy.

**How to use (Colab)**
1. Put the D2 export (file or folder of parquet parts) in the same Drive folder as D1. D1's `cluster_id` column is read to reuse the *same* LOCO folds as notebook A.
2. Run notebook A first if you want the composition section at the end: it needs `results/modelA_A3_tau0.90.json` and `results/modelA_meta.json`.
3. Set `DATA_DIR`, then **Runtime > Run all**. CPU runtime is fine.

**Conventions.** In this notebook `y = 1` means **NOT ready** (the risk being predicted), so PR-AUC is about the failure class.
Train/test negative rate is ~46% but validation is ~36% (plan 4.6). Isotonic calibration fitted on validation therefore imports that shift; the temporal
section compares *uncalibrated*, *validation-calibrated* and *out-of-fold (train) calibrated* versions, so you can see the effect before choosing.

**Budgets.** Free Colab has 2 vCPUs. Training rows per fit are capped (`B_TRAIN_MAX_ROWS`, `RF_MAX_SAMPLES`) so the 10-fold LOCO finishes in roughly an hour.
Set them to `None` for full-data fits if you have time (Colab Pro, or a local machine).

**Tested on** a synthetic stand-in with the plan's schema, not on your real data.

## 1. Setup and configuration
Run this cell first. It mounts Google Drive (in Colab), installs anything missing, and holds **every setting you may want to change**.
Nothing below this block should need editing unless a check cell tells you a column name differs from the plan.

In [ ]:
import os, sys, glob, fnmatch, subprocess, warnings, time, json, math
warnings.filterwarnings("ignore")
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:                                       # Colab already ships most of these; this only adds/upgrades what is missing
    import importlib.util                          # install only what is missing, so pip never swaps a library under a running session
    _need = [p for p, m in (("xgboost>=2.0", "xgboost"), ("lightgbm", "lightgbm"), ("shap", "shap")) if importlib.util.find_spec(m) is None]
    if _need: subprocess.run([sys.executable, "-m", "pip", "install", "-q", *_need], check=False)
    from google.colab import drive
    drive.mount("/content/drive")
import numpy as np, pandas as pd, pyarrow as pa, pyarrow.parquet as pq
import matplotlib
try: get_ipython(); _IN_NOTEBOOK = True          # Colab / Jupyter: keep the inline backend so figures show
except NameError: _IN_NOTEBOOK = False; matplotlib.use("Agg")   # plain-python run: draw to files only
import matplotlib.pyplot as plt
import xgboost as xgb
from scipy import stats

# ------------------------------ CONFIG ------------------------------
FAST = os.environ.get("ISAN_FAST", "0") == "1"      # 1 = smoke-test profile with tiny budgets. Leave 0 for real runs.
DATA_DIR = os.environ.get("ISAN_DATA_DIR", "/content/drive/MyDrive/ISAN3392")   # folder holding the D1 / D2 parquet exports
OUT_DIR = os.environ.get("ISAN_OUT_DIR", os.path.join(DATA_DIR, "results"))     # models, tables and figures are written here
D1_PATTERN, D2_PATTERN = "*D1*", "*D2*"             # name patterns (file or folder, case-insensitive) inside DATA_DIR
SEED = 3392
TRAIN_DAYS, VAL_DAYS, TEST_DAYS = (14, 129), (130, 157), (158, 183)      # plan section 4.1 (admission day)
MIN_CLUSTER_PODS = 1000 if FAST else 9000           # plan 4.3: clusters with >= 9,000 eligible D1 pods become LOCO folds
ALPHA = 0.05                                        # conformal miscoverage. 0.05 matches the headline U <= 5% (the plan's 90% does not)
TAU_MAIN = 0.90                                     # operating quantile of the base model
HIST_WINDOWS = (3, 7, 14)                           # A1 sliding-window lengths in days
N_JOBS = -1                                         # all cores
FIG_DPI = 110
os.makedirs(OUT_DIR, exist_ok=True)
rng = np.random.default_rng(SEED)
print(f"FAST={FAST} | IN_COLAB={IN_COLAB} | DATA_DIR={DATA_DIR}")
print("files found:", sorted(os.listdir(DATA_DIR)) if os.path.isdir(DATA_DIR) else "DATA_DIR NOT FOUND - fix DATA_DIR above")
print("xgboost", xgb.__version__, "| pandas", pd.__version__, "| numpy", np.__version__)
assert tuple(int(x) for x in xgb.__version__.split(".")[:2]) >= (2, 0), "need xgboost >= 2.0 for reg:quantileerror"

## 2. Schema, leakage gate and loader
The feature lists below are copied from plan sections 3.3 and 3.4. **If your real export uses different column names,
the schema report printed when the data is loaded will show exactly which ones, and you only change the lists here.**

In [ ]:
NUMERIC = ["gpu_request", "vram_entitled_gib", "cpu_request_cores", "server_gpu_count", "server_cpu_capacity_cores",
           "gpu_mem_per_gpu", "cpu_per_gpu", "adm_hour", "adm_dow_idx", "hour_sin", "hour_cos", "dow_sin", "dow_cos",
           "is_whole_gpu", "is_fractional", "is_multi_gpu", "has_workload_id",
           "cluster_pods_at_submit", "cluster_gpu_requested_at_submit", "cluster_sm_mean_prev_hr", "cluster_sm_p90_prev_hr",
           "server_alloc_gpu_at_submit", "server_gpu_occupancy_at_submit", "asw_pods_at_submit", "asw_alloc_gpu_at_submit",
           "wl_hist_pod_count", "wl_hist_days_observed", "wl_hist_mvf_p90", "wl_hist_mvf_max", "wl_hist_mvf_mean",
           "wl_hist_sm_p90", "wl_hist_dur_mean", "is_genai_request"]
CATEG = ["gpu_spec_public", "priority_class", "model_type_public"]     # categoricals that transfer across clusters
CATEG_CLUSTER_SPECIFIC = ["cluster_id", "asw_id"]                       # cannot generalise to an unseen cluster -> excluded from LOCO models
HIST_COLS = [c for c in NUMERIC if c.startswith("wl_hist_")] + ["has_workload_id"]
CLUSTER_LOAD_COLS = ["cluster_pods_at_submit", "cluster_gpu_requested_at_submit", "cluster_sm_mean_prev_hr", "cluster_sm_p90_prev_hr"]
# Plan 3.4: derived from observed utilisation -> a build gate fails if any of these is used as a predictor
LEAKY = {"avg_gpu_sm_util", "avg_gpu_mem_gib", "used_gpu_hours", "sm_p90_norm", "sm_mean_norm", "mem_frac_p90",
         "peak_to_mean", "waste_tier", "wasted_gpu_hours", "min_viable_fraction"}
UNKNOWN_AT_ADMISSION = {"n_pod_hours", "n_hours_with_util", "is_censored", "schedule_delay_sec", "ready_delay_sec", "duration_hours"}
TARGETS = {"y_mvf", "y_ready", "y_ready_jes_crosscheck"}
# Monotone constraints (A3). Only the history features: "more historical demand never lowers the slice" is a property
# you can state in one sentence. cpu_per_gpu is NOT constrained: the OSDI'26 paper (3.4) shows CTR/rec pods are CPU-heavy but low-SM,
# so its sign is probably negative across segments. Section 7 prints the within-segment sign so you can decide with evidence.
MONO = {"wl_hist_mvf_p90": 1, "wl_hist_mvf_max": 1, "wl_hist_mvf_mean": 1, "wl_hist_sm_p90": 1}
DAY_CANDIDATES = ["adm_day", "admission_day", "first_day", "day"]
DUR_CANDIDATES = ["duration_hours", "n_pod_hours", "n_hours_with_util", "duration_h", "duration_hrs", "observed_hours", "duration"]   # how long each pod was observed (hours)
WL_CANDIDATES = ["workload_code", "workload_id_code", "wl_code", "workload_id"]

def _resolve(path_cols, wanted):
    low = {c.lower(): c for c in path_cols}
    return [low[w.lower()] for w in wanted if w.lower() in low]

def read_parquet_any(path, columns=None):
    """Read a parquet file or a folder of parts. DECIMAL -> float64 at the Arrow level, before pandas ever sees it
    (Snowflake NUMBER otherwise arrives as Python Decimal objects: 13.6x more RAM, see plan 6.5). Column names are lower-cased."""
    files = sorted(glob.glob(os.path.join(path, "**", "*.parquet"), recursive=True)) if os.path.isdir(path) else [path]
    if not files: raise FileNotFoundError(f"no .parquet under {path}")
    parts = []
    for f in files:
        cols = _resolve(pq.read_schema(f).names, columns) if columns else None
        t = pq.read_table(f, columns=cols)
        for i, fld in enumerate(t.schema):
            if pa.types.is_decimal(fld.type): t = t.set_column(i, fld.name, t.column(i).cast(pa.float64()))
        d = t.to_pandas(); d.columns = [c.lower() for c in d.columns]; parts.append(d)
    return pd.concat(parts, ignore_index=True)

def find_path(pattern):
    hits = [p for p in sorted(os.listdir(DATA_DIR)) if fnmatch.fnmatch(p.lower(), pattern.lower())
            and (p.lower().endswith(".parquet") or os.path.isdir(os.path.join(DATA_DIR, p)))]      # skip dictionaries, loaders, csv notes
    if not hits: raise FileNotFoundError(f"nothing matching {pattern!r} in {DATA_DIR}. Found: {sorted(os.listdir(DATA_DIR))}")
    return os.path.join(DATA_DIR, hits[0])

def pick_col(df, candidates, what):
    for c in candidates:
        if c in df.columns: return c
    raise KeyError(f"cannot find the {what} column. Tried {candidates}. Columns present: {list(df.columns)}")
print(f"{len(NUMERIC)} numeric + {len(CATEG)} transferable categorical features; {len(LEAKY)} leaky and {len(UNKNOWN_AT_ADMISSION)} unknown-at-admission columns are barred")

## 3. Data-preparation helpers: `prepare`, schema report, leakage gate
These functions are called when each dataset is loaded below. `prepare()` does what plan 6.5 asks of the loader (Decimal -> float, categoricals, int group codes) and adds the helper columns
used everywhere else. **Read the printed report**: it lists plan columns that are missing and columns the plan does not mention.

In [ ]:
GROUP_FOR_MISSING = "shape"   # pods with NO workload_id: "shape" = group near-duplicates by resource-request signature, "unique" = one group per pod
                              # (never give them all one shared code: GroupKFold would put 67% of the data in a single fold, see review item 4)

def prepare(df, target, name):
    df = df.copy()
    day_col = pick_col(df, DAY_CANDIDATES, "admission-day")
    wl_col = pick_col(df, WL_CANDIDATES, "workload id / code")
    tgt = pick_col(df, [target], "target")
    df["adm_day_"] = df[day_col].astype("int32")
    s = df[wl_col]
    if pd.api.types.is_numeric_dtype(s): s = s.where(s >= 0)           # numeric codes: negative = missing
    codes = pd.factorize(s)[0].astype("int64")                         # -1 where missing
    has = codes >= 0
    n = len(df)
    if GROUP_FOR_MISSING == "unique":
        miss_group = codes.max() + 1 + np.arange(n)
    else:
        key = df[[c for c in ["cluster_id", "gpu_spec_public", "gpu_request", "cpu_request_cores", "vram_entitled_gib", "priority_class", "model_type_public"] if c in df.columns]]
        miss_group = codes.max() + 1 + (pd.util.hash_pandas_object(key, index=False).values % 10**9).astype("int64")
    df["wl_code"] = codes; df["group"] = np.where(has, codes, miss_group)
    df["t_adm"] = df["adm_day_"].astype("float64") * 24 + df["adm_hour"].astype("float64")
    dur_col = next((c for c in DUR_CANDIDATES if c in df.columns), None)
    if dur_col:
        df["obs_hours"] = df[dur_col].astype("float64").fillna(24.0); print(f"  pod duration taken from column '{dur_col}'")
    else:
        df["obs_hours"] = 24.0
        print("  WARNING: no duration column found (tried %s).\n"
              "           Assuming every pod ran 24 h (the D1 minimum). The A1 history and the leakage audit are then approximate:\n"
              "           re-export D1/D2 with duration_hours (or n_pod_hours) for exact results." % DUR_CANDIDATES)
    df["t_end"] = df["t_adm"] + df["obs_hours"]                       # when the pod's label became complete
    if "reserved_gpu_hours" not in df.columns:
        df["reserved_gpu_hours"] = df["gpu_request"].astype("float64") * df["obs_hours"]
        print("  note: reserved_gpu_hours not in export -> computed as gpu_request * observed hours")
    for c in CATEG + CATEG_CLUSTER_SPECIFIC:
        if c in df.columns: df[c] = df[c].astype(str).astype("category")
    for c in NUMERIC:
        if c in df.columns: df[c] = df[c].astype("float32")
    df["y"] = df[tgt].astype("float64")
    df["segment"] = np.select([df["adm_day_"] < TRAIN_DAYS[0], df["adm_day_"] <= TRAIN_DAYS[1], df["adm_day_"] <= VAL_DAYS[1], df["adm_day_"] <= TEST_DAYS[1]],
                              ["warmup", "train", "val", "test"], "tail")
    df["cold"] = df["wl_hist_pod_count"].isna() if "wl_hist_pod_count" in df.columns else True
    df["is_rec"] = (df["model_type_public"].astype(str) == "rec")
    df["stratum"] = np.where(df["is_rec"], "rec", "genai") + "|" + np.where(df["cold"], "cold", "warm")
    print(f"{name}: {df.shape[0]:,} rows x {df.shape[1]} cols | workload id present {has.mean():.1%} | largest group = {np.bincount(pd.factorize(df['group'])[0]).max() / n:.2%} of rows")
    return df

def schema_report(df):
    global NUM_F, CAT_F, CLS_F
    NUM_F = [c for c in NUMERIC if c in df.columns]; CAT_F = [c for c in CATEG if c in df.columns]
    CLS_F = [c for c in CATEG_CLUSTER_SPECIFIC if c in df.columns]
    miss = [c for c in NUMERIC + CATEG if c not in df.columns]
    known = set(NUMERIC) | set(CATEG) | set(CATEG_CLUSTER_SPECIFIC) | LEAKY | UNKNOWN_AT_ADMISSION | TARGETS | set(DAY_CANDIDATES) | set(WL_CANDIDATES)
    helper = {"adm_day_", "wl_code", "group", "t_adm", "t_end", "obs_hours", "y", "segment", "cold", "is_rec", "stratum", "reserved_gpu_hours", "pod_id"}
    unk = [c for c in df.columns if c not in known and c not in helper]
    print(f"features available: {len(NUM_F)} numeric + {len(CAT_F)} categorical (+ cluster-specific: {CLS_F})")
    if miss: print("!! plan columns NOT in your export (dropped from the feature lists):", miss)
    if unk: print("?? columns the plan does not classify (NOT used - add to NUMERIC/CATEG only if they are known at admission):", unk)

def leakage_gate(features, df, target_col="y", sample=200_000):
    bad = (set(features) & (LEAKY | UNKNOWN_AT_ADMISSION | TARGETS))
    assert not bad, f"LEAKAGE GATE FAILED - barred columns used as predictors: {sorted(bad)}"
    s = df.sample(min(sample, len(df)), random_state=SEED); hard, soft = [], []
    for c in features:
        if str(s[c].dtype) == "category": continue
        v = s[[c, target_col]].dropna()
        if len(v) > 500 and v[c].nunique() > 1:
            r = abs(stats.spearmanr(v[c], v[target_col])[0])
            if r > 0.97: hard.append((c, round(float(r), 3)))
            elif r > 0.85: soft.append((c, round(float(r), 3)))
    assert not hard, f"SINGLE-FEATURE LEAKAGE SCREEN: |Spearman| > 0.97 with the target for {hard} - almost certainly a leaked column, inspect before training"
    if soft: print("  warning: very strong single features (|Spearman| 0.85-0.97):", soft, "- expected for wl_hist_* on warm rows, but this is exactly where leakage hides: see the A1 history audit in notebook A")
    print(f"leakage gate passed for {len(features)} features (barred-column check + single-feature screen)")

## 4. Splits: temporal, grouped, leave-one-cluster-out (LOCO)
* **Temporal** (plan 4.1): train = days 14-129, validation = 130-157, test = 158-183, by admission day.
* The validation segment is split into two **workload-disjoint halves** (by group parity): one for early stopping / model selection,
  one for conformal or isotonic calibration. Using the same rows for both would make the calibration optimistic.
* **LOCO protocol used throughout** (strict version): for held-out cluster *k* the model is trained on the *other* clusters' train days,
  tuned and calibrated on the *other* clusters' validation days, and scored on cluster *k*'s **test days only**. It is therefore unseen in
  both cluster and time. Hyper-parameters are chosen once on the pooled temporal validation fold and reused (equal budget, no per-fold tuning).

In [ ]:
def make_masks(df):
    seg = df["segment"].values; par = (df["group"].values % 2) == 0
    m = {s: seg == s for s in ("train", "val", "test")}
    m["val_es"] = m["val"] & par; m["val_cal"] = m["val"] & ~par
    return m

def eligible_clusters(cluster_series):
    vc = cluster_series.astype(str).value_counts()
    return [c for c in vc.index if vc[c] >= MIN_CLUSTER_PODS]

def loco_masks(df, masks, cluster):
    other = (df["cluster_id"].astype(str) != cluster).values; this = ~other
    return dict(train=masks["train"] & other, es=masks["val_es"] & other, cal=masks["val_cal"] & other, test=masks["test"] & this)

def fold_table(df, clusters):
    rows = []
    for c in clusters:
        a = df[df["cluster_id"].astype(str) == c]; t = a[a["segment"] == "test"]
        rows.append(dict(cluster=c, n_all=len(a), n_test=len(t), rec_share=round(float(a["is_rec"].mean()), 3),
                         y_p50=round(float(a["y"].median()), 3), warm_share_test=round(float(1 - t["cold"].mean()), 3) if len(t) else np.nan))
    return pd.DataFrame(rows).set_index("cluster")

## 5. Metrics and the conformal / slice-snapping machinery
* **Pinball loss** at tau, **empirical coverage**, and the plan's two operational numbers: **U** (share of pods whose recommended slice is below
  their label) and **R** (reclaimed GPU-hours). R is reported as a **share of reserved GPU-hours** and computed as `reserved_hours x (1 - slice_fraction)`,
  which fixes the double-counted request units in plan 6.1.
* One-sided **split-conformal** margin: `q = the ceil((n+1)(1-alpha))/n quantile of (y - yhat)` on the calibration half, then `slice = snap_up(yhat + q)`.
  A **Mondrian** (stratified) version calibrates separately per model_type x cold/warm, because one global margin over-covers easy strata and under-covers hard ones.
  Split-conformal assumes exchangeability, which temporal and cluster shift break - so coverage on unseen clusters is something we *measure*, not a guarantee.

In [ ]:
def pinball(y, p, tau):
    d = np.asarray(y) - np.asarray(p); return float(np.mean(np.maximum(tau * d, (tau - 1) * d)))

def conformal_margin(y_cal, p_cal, alpha):
    r = np.sort(np.asarray(y_cal) - np.asarray(p_cal)); n = len(r)
    if n == 0: return 0.0
    k = int(math.ceil((n + 1) * (1 - alpha))); return float(r[min(k, n) - 1])

def mondrian_margins(y_cal, p_cal, strata_cal, alpha, min_n=200):
    g = conformal_margin(y_cal, p_cal, alpha); out = {}
    strata_cal = np.asarray(strata_cal)
    for s in np.unique(strata_cal):
        m = strata_cal == s; out[s] = conformal_margin(y_cal[m], p_cal[m], alpha) if m.sum() >= min_n else g
    return g, out

def upper_bound(p, strata=None, g=0.0, per_stratum=None):
    q = np.full(len(p), g, dtype="float64")
    if per_stratum is not None and strata is not None: q = pd.Series(np.asarray(strata)).map(per_stratum).fillna(g).values
    return np.clip(np.asarray(p) + q, 0, 1)

def slice_fraction(p_up, req, ladder):
    """snap_up: smallest deployable size >= predicted need (always upward), never above the original request."""
    req = np.asarray(req, dtype="float64"); v = np.clip(p_up, 0, 1) * req
    i = np.minimum(np.searchsorted(ladder, v - 1e-9, side="left"), len(ladder) - 1)
    s = np.where(v > ladder[-1] + 1e-9, req, ladder[i]); return np.minimum(s, req) / req

def operating_point(y, req, res_hours, p_up, ladder):
    sf = slice_fraction(p_up, req, ladder)
    return dict(U=float(np.mean(sf < np.asarray(y) - 1e-9)), R_share=float(np.sum(res_hours * (1 - sf)) / np.sum(res_hours)), mean_slice=float(np.mean(sf)))

def frontier(y, req, res, p_base, ladder, margins=None):
    margins = np.linspace(-0.3, 0.7, 51) if margins is None else margins
    pts = [operating_point(y, req, res, np.clip(p_base + m, 0, 1), ladder) for m in margins]
    return pd.DataFrame(pts).assign(margin=margins)

def r_at_u(fr, u_max):
    """Largest reclaimed share among frontier points with U <= u_max (0 if none qualify)."""
    ok = fr[fr["U"] <= u_max]; return float(ok["R_share"].max()) if len(ok) else 0.0

def paired_test(a, b, alternative="greater"):
    """Wilcoxon signed-rank on paired per-fold scores (exact for n <= 25)."""
    a, b = np.asarray(a, float), np.asarray(b, float); d = a - b
    if np.allclose(d, 0): return dict(W=np.nan, p=1.0, n=len(d))
    r = stats.wilcoxon(a, b, alternative=alternative, zero_method="wilcox"); return dict(W=float(r.statistic), p=float(r.pvalue), n=int(len(d)))
print("metrics ready: pinball, conformal_margin, mondrian_margins, upper_bound, slice_fraction, operating_point, frontier, r_at_u, paired_test")

## 6. Design matrix for the linear models (shared by A2 and B1)
Median imputation + a missing-indicator per column that has NaNs (structural missingness carries signal), standardisation, one-hot categoricals with fixed levels.

In [ ]:
class Design:
    def __init__(self, num, cat): self.num, self.cat = list(num), list(cat)
    def fit(self, d):
        X = d[self.num].astype("float64"); self.med = X.median().fillna(0.0)
        self.na_cols = [c for c in self.num if X[c].isna().any()]
        Xi = X.fillna(self.med); self.mu = Xi.mean(); self.sd = Xi.std().replace(0, 1).fillna(1.0)
        self.levels = {c: np.array(list(d[c].cat.categories), dtype=object) for c in self.cat}; return self
    def transform(self, d, dtype="float64"):
        X = d[self.num].astype("float64"); na = X[self.na_cols].isna().astype("float64").values
        Z = ((X.fillna(self.med) - self.mu) / self.sd).clip(-5, 5).values
        oh = [(d[c].astype(str).to_numpy(dtype=object)[:, None] == self.levels[c][None, :]).astype("float64") for c in self.cat]
        return np.hstack([Z, na] + oh).astype(dtype, copy=False)
print("Design matrix helper ready")

## 7. Load D2, schema report, folds
Folds are the clusters that qualify in **D1** (>= `MIN_CLUSTER_PODS` eligible pods), so A and B are tested on the same held-out clusters.

In [ ]:
from IPython.display import display
import lightgbm as lgb
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.isotonic import IsotonicRegression
from sklearn.metrics import average_precision_score, brier_score_loss, log_loss, roc_auc_score
from sklearn.model_selection import GroupKFold

t0 = time.time()
df = prepare(read_parquet_any(find_path(D2_PATTERN)), "y_ready", "D2")
df["y"] = 1.0 - df["y"]                                  # y = 1 -> pod did NOT become ready (the event we predict)
schema_report(df)
df = df[df["segment"].isin(["train", "val", "test"])].reset_index(drop=True)
FEATS = NUM_F + CAT_F
leakage_gate(FEATS, df)
masks = make_masks(df)
try:
    FOLDS = eligible_clusters(read_parquet_any(find_path(D1_PATTERN), columns=["cluster_id"])["cluster_id"]); print("folds taken from D1 eligibility")
except Exception as e:
    vc = df["cluster_id"].astype(str).value_counts(); FOLDS = list(vc.index[:10]); print(f"D1 not readable ({type(e).__name__}); using the 10 largest D2 clusters")
if FAST: FOLDS = FOLDS[:4]
print(f"loaded in {time.time() - t0:.0f}s | {len(FOLDS)} LOCO folds: {FOLDS}")
print("not-ready rate by segment:", {k: round(float(df.loc[masks[k], "y"].mean()), 3) for k in ("train", "val", "test")})
rows = []
for c in FOLDS:
    a = df[df["cluster_id"].astype(str) == c]; t = a[a["segment"] == "test"]
    rows.append(dict(cluster=c, n_all=len(a), n_test=len(t), rec_share=round(float(a["is_rec"].mean()), 3), fail_rate_test=round(float(t["y"].mean()), 3),
                     fractional_share=round(float((a["gpu_request"] < 1).mean()), 3)))
FT = pd.DataFrame(rows).set_index("cluster"); display(FT)
B_TRAIN_MAX_ROWS = 60_000 if FAST else 800_000            # rows per training fit (random subsample); None = all
RF_TREES, RF_MAX_SAMPLES = (40, 20_000) if FAST else (150, 150_000)
def cap(mask, n=B_TRAIN_MAX_ROWS):
    idx = np.flatnonzero(mask)
    if n is not None and len(idx) > n: idx = np.sort(np.random.default_rng(SEED).choice(idx, n, replace=False))
    return idx

## 8. The three families, calibration, and metrics
* **B1** L2 logistic regression on the shared design matrix (+ Platt scaling) - the linear, interpretable, calibrated floor.
* **B2** random forest, no class weighting (classes are naturally balanced) - does bagging alone suffice?
* **B3** LightGBM `binary` with native categoricals and early stopping (+ isotonic calibration) - does boosting add discrimination *and* calibration?

Metrics: **PR-AUC** (failure class), ROC-AUC, **Brier**, log loss, **ECE** (15 equal-width bins). Selection uses log loss on the held-out validation half
(a proper scoring rule), three configurations per family.

In [ ]:
N_EST_B, ES_B = (200, 20) if FAST else (3000, 50)
def _logit(p): p = np.clip(p, 1e-6, 1 - 1e-6); return np.log(p / (1 - p))

class Calibrator:
    def __init__(self, method): self.method = method
    def fit(self, p, y):
        if self.method == "isotonic": self.f = IsotonicRegression(out_of_bounds="clip", y_min=0, y_max=1).fit(p, y)
        elif self.method == "platt": self.f = LogisticRegression(C=1e6).fit(_logit(p)[:, None], y)
        return self
    def transform(self, p):
        if self.method == "none": return np.asarray(p)
        return self.f.predict(p) if self.method == "isotonic" else self.f.predict_proba(_logit(p)[:, None])[:, 1]

def ece(y, p, bins=15):
    y, p = np.asarray(y), np.asarray(p); b = np.clip(np.digitize(p, np.linspace(0, 1, bins + 1)[1:-1]), 0, bins - 1); tot = 0.0
    for k in range(bins):
        m = b == k
        if m.any(): tot += m.mean() * abs(p[m].mean() - y[m].mean())
    return float(tot)

def bmetrics(y, p):
    p = np.clip(p, 1e-6, 1 - 1e-6)
    return dict(pr_auc=average_precision_score(y, p), roc_auc=roc_auc_score(y, p), brier=brier_score_loss(y, p), logloss=log_loss(y, p), ece=ece(y, p))

def tree_X(d):
    X = d[NUM_F].astype("float32").fillna(-1.0)
    for c in CAT_F: X[c] = d[c].cat.codes.astype("int16")
    return X

class B1:
    name, cal_method = "B1", "platt"
    def __init__(self, C): self.C = C
    def fit(self, dtr, des=None):
        self.des = Design(NUM_F, CAT_F).fit(dtr); self.m = LogisticRegression(C=self.C, max_iter=500).fit(self.des.transform(dtr, "float32"), dtr["y"].values); return self
    def predict(self, d): return self.m.predict_proba(self.des.transform(d, "float32"))[:, 1]

class B2:
    name, cal_method = "B2", "isotonic"
    def __init__(self, leaf): self.leaf = leaf
    def fit(self, dtr, des=None):
        self.m = RandomForestClassifier(n_estimators=RF_TREES, min_samples_leaf=self.leaf, max_features="sqrt", max_samples=min(RF_MAX_SAMPLES, len(dtr)),
                                        n_jobs=N_JOBS, random_state=SEED).fit(tree_X(dtr), dtr["y"].values); return self
    def predict(self, d): return self.m.predict_proba(tree_X(d))[:, 1]

class B3:
    name, cal_method = "B3", "isotonic"
    def __init__(self, leaves): self.leaves = leaves
    def fit(self, dtr, des):
        self.m = lgb.LGBMClassifier(objective="binary", n_estimators=N_EST_B, learning_rate=0.05, num_leaves=self.leaves, min_child_samples=100,
                                    subsample=0.8, subsample_freq=1, colsample_bytree=0.8, n_jobs=N_JOBS, random_state=SEED, verbose=-1)
        self.m.fit(dtr[FEATS], dtr["y"].values, eval_set=[(des[FEATS], des["y"].values)], callbacks=[lgb.early_stopping(ES_B, verbose=False)]); return self
    def predict(self, d): return self.m.predict_proba(d[FEATS])[:, 1]

GRIDS = {"B1": (B1, (0.01, 0.1, 1.0)), "B2": (B2, (20, 50, 200)), "B3": (B3, (31, 63, 127))}
BEST, rows = {}, []
tr_idx, es_d, sel_d = cap(masks["train"]), df[masks["val_es"]], df[masks["val_cal"]]
for name, (cls, grid) in GRIDS.items():
    best = None
    for g in grid:
        t0 = time.time(); m = cls(g).fit(df.iloc[tr_idx], es_d); ll = bmetrics(sel_d["y"].values, m.predict(sel_d))["logloss"]
        rows.append(dict(model=name, config=g, sel_logloss=round(ll, 5), seconds=round(time.time() - t0, 1)))
        if best is None or ll < best[0]: best = (ll, g)
    BEST[name] = best[1]
display(pd.DataFrame(rows)); print("selected:", BEST)

## 9. Temporal evaluation and the calibration-source diagnostic
Each family is refit with its selected configuration, then scored on the test days **raw**, after calibration on the validation half (**val-cal**),
and - for B3 - after isotonic calibration on **grouped out-of-fold predictions from the training days** (**oof-cal**). Because the validation window's
failure rate (~36%) differs from train/test (~46%), compare the ECE of val-cal and oof-cal before choosing a calibration source for the write-up.

In [ ]:
TB, rows, REL = {}, [], {}
dte = df[masks["test"]]; yt = dte["y"].values
for name, (cls, _) in GRIDS.items():
    m = cls(BEST[name]).fit(df.iloc[tr_idx], es_d); cal = Calibrator(cls.cal_method).fit(m.predict(sel_d), sel_d["y"].values)
    pt = m.predict(dte); TB[name], TB[name + "_cal"] = m, cal
    for tag, p in (("raw", pt), ("val-cal", cal.transform(pt))):
        rows.append(dict(model=name, calibration=tag, **bmetrics(yt, p))); REL[(name, tag)] = p
oof_idx = cap(masks["train"], 30_000 if FAST else 300_000); dtr_o = df.iloc[oof_idx]; oof = np.zeros(len(dtr_o))
for a, b in GroupKFold(3).split(dtr_o, groups=dtr_o["group"].values):
    oof[b] = B3(BEST["B3"]).fit(dtr_o.iloc[a], es_d).predict(dtr_o.iloc[b])
CAL_OOF = Calibrator("isotonic").fit(oof, dtr_o["y"].values); p_oof = CAL_OOF.transform(TB["B3"].predict(dte)); REL[("B3", "oof-cal")] = p_oof
rows.append(dict(model="B3", calibration="oof-cal", **bmetrics(yt, p_oof)))
BT = pd.DataFrame(rows).set_index(["model", "calibration"]).round(4); BT.to_csv(os.path.join(OUT_DIR, "B_temporal_results.csv")); display(BT)
print(f"test failure rate {yt.mean():.3f} | validation-half failure rate {sel_d['y'].mean():.3f} | train failure rate {df.loc[masks['train'], 'y'].mean():.3f}")

def reliability(ax, y, p, label, bins=10):
    e = np.linspace(0, 1, bins + 1); b = np.clip(np.digitize(p, e[1:-1]), 0, bins - 1)
    xs = [p[b == k].mean() for k in range(bins) if (b == k).sum() >= 20]; ys = [y[b == k].mean() for k in range(bins) if (b == k).sum() >= 20]
    ax.plot(xs, ys, marker="o", label=f"{label} (ECE {ece(y, p):.3f})")
fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))
for name in GRIDS: reliability(ax[0], yt, REL[(name, "val-cal")], f"{name} val-cal")
for tag in ("raw", "val-cal", "oof-cal"): reliability(ax[1], yt, REL[("B3", tag)], f"B3 {tag}")
for a, t in zip(ax, ("Families after validation calibration", "B3: which calibration source?")):
    a.plot([0, 1], [0, 1], "k:", lw=1); a.set_xlabel("predicted P(not ready)"); a.set_ylabel("observed frequency"); a.set_title(t); a.legend(fontsize=8)
plt.tight_layout(); plt.savefig(os.path.join(OUT_DIR, "B_reliability.png"), dpi=FIG_DPI); plt.show()

## 10. Leave-one-cluster-out for B1, B2, B3 and the paired tests
Same strict protocol as notebook A: train on other clusters' train days (capped), early-stop on their validation half *es*, calibrate on their half *cal*,
score the held-out cluster's test days. Primary pre-registered comparison: **B3 vs B1** on PR-AUC and on ECE (one-sided Wilcoxon over folds);
B3 vs B2 is secondary.

In [ ]:
LB, t_start = [], time.time()
for k, c in enumerate(FOLDS, 1):
    lm = loco_masks(df, masks, c); t0 = time.time(); d_te, d_es, d_cal = df[lm["test"]], df[lm["es"]], df[lm["cal"]]; idx = cap(lm["train"])
    for name, (cls, _) in GRIDS.items():
        m = cls(BEST[name]).fit(df.iloc[idx], d_es); cal = Calibrator(cls.cal_method).fit(m.predict(d_cal), d_cal["y"].values); pt = m.predict(d_te)
        for tag, p in (("raw", pt), ("cal", cal.transform(pt))): LB.append(dict(model=name, fold=c, calibration=tag, n_test=len(d_te), **bmetrics(d_te["y"].values, p)))
    el = time.time() - t0; print(f"fold {k}/{len(FOLDS)} {c}: {el:.0f}s (est. remaining {el * (len(FOLDS) - k) / 60:.1f} min)")
LB = pd.DataFrame(LB); LB.to_csv(os.path.join(OUT_DIR, "B_loco_results.csv"), index=False)
display(LB.groupby(["model", "calibration"])[["pr_auc", "roc_auc", "brier", "ece"]].agg(["mean", "std"]).round(4))
C = LB[LB.calibration == "cal"].pivot(index="fold", columns="model", values=["pr_auc", "ece"])
t1 = paired_test(C[("pr_auc", "B3")], C[("pr_auc", "B1")], "greater"); t2 = paired_test(C[("ece", "B1")], C[("ece", "B3")], "greater")
t3 = paired_test(C[("pr_auc", "B3")], C[("pr_auc", "B2")], "greater")
print(f"B3 vs B1  PR-AUC higher: wins {int((C[('pr_auc','B3')] > C[('pr_auc','B1')]).sum())}/{len(C)}, Wilcoxon p = {t1['p']:.4f}")
print(f"B3 vs B1  ECE lower:     wins {int((C[('ece','B3')] < C[('ece','B1')]).sum())}/{len(C)}, Wilcoxon p = {t2['p']:.4f}")
print(f"B3 vs B2  PR-AUC higher (secondary): Wilcoxon p = {t3['p']:.4f}  (smallest possible with {len(C)} folds: {1 / 2 ** len(C):.4f})")

## 11. B3 attribution, latency, saved artefacts
Split-gain importance and SHAP contributions (LightGBM's built-in `pred_contrib`, in log-odds) for the temporal B3, single-call latency,
and the files the composition section (and any later scheduler prototype) loads.

In [ ]:
import pickle
B3T = TB["B3"]; bst = B3T.m.booster_
gain_b = pd.Series(bst.feature_importance("gain"), index=bst.feature_name()).sort_values(ascending=False)
Xs_b = dte[FEATS].sample(min(5000, len(dte)), random_state=SEED)
contrib = B3T.m.predict(Xs_b, pred_contrib=True)
fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))
gain_b.head(15)[::-1].plot.barh(ax=ax[0], title="B3 split gain")
pd.Series(np.abs(contrib[:, :-1]).mean(0), index=FEATS).sort_values().tail(15).plot.barh(ax=ax[1], title="B3 mean |SHAP| (log-odds)")
plt.tight_layout(); plt.savefig(os.path.join(OUT_DIR, "B_importance.png"), dpi=FIG_DPI); plt.show()
lat = []
for i in range(300):
    r = dte[FEATS].iloc[[i % len(dte)]]; t0 = time.perf_counter(); B3T.m.predict_proba(r); lat.append((time.perf_counter() - t0) * 1e3)
print(f"B3 single-call latency: p50 {np.percentile(lat, 50):.2f} ms | p95 {np.percentile(lat, 95):.2f} ms | trees used: {B3T.m.best_iteration_ or N_EST_B}")
bst.save_model(os.path.join(OUT_DIR, "modelB_B3.txt"))
pickle.dump(dict(val=TB["B3_cal"], oof=CAL_OOF), open(os.path.join(OUT_DIR, "modelB_B3_calibrators.pkl"), "wb"))
print("saved:", sorted(f for f in os.listdir(OUT_DIR) if f.startswith(("B_", "modelB"))))

## 12. RQ3: composing A and B into a sizing policy (what-if analysis)
For each test pod requesting >= 1 GPU: Model A recommends a slice (A3 + Mondrian conformal margin + snap-up); Model B is queried twice, at the
**original** request and at the **recommended** one (request-dependent features recomputed), giving the change in failure risk the shrink would cause.
A policy shrinks a pod only when that change is below `RISK_MAX`.

**Read with care.** Querying B at a size the pod did not request is a counterfactual: B learned P(not ready | request, ...) from *observational* data,
so the shift is only as trustworthy as the overlap between what-if sizes and sizes actually seen. The **positivity** column reports the share of what-if
rows whose (GPU model, new size) occurs >= 30 times in training. U (under-provisioning) cannot be measured here - D2 has no utilisation labels -
so the safety side of the trade-off comes from notebook A's held-out evaluation.

In [ ]:
CAL_FOR_POLICY = "oof"          # "oof" or "val": pick using the reliability diagnostic above
pA_path, pM_path = os.path.join(OUT_DIR, "modelA_A3_tau0.90.json"), os.path.join(OUT_DIR, "modelA_meta.json")
if not (os.path.exists(pA_path) and os.path.exists(pM_path)):
    print("Model A artefacts not found in OUT_DIR - run notebook A first, then rerun this cell.")
else:
    meta = json.load(open(pM_path)); MA = xgb.XGBRegressor(); MA.load_model(pA_path); LAD = np.array(meta["ladder"])
    cal = CAL_OOF if CAL_FOR_POLICY == "oof" else TB["B3_cal"]
    d = dte[dte["gpu_request"] >= 1].copy(); XA = d[meta["features"]].copy()
    for c in meta["categorical"]: XA[c] = pd.Categorical(XA[c].astype(str), categories=meta["categories"][c])
    up = upper_bound(np.clip(MA.predict(XA), 0, 1), d["stratum"].values, meta["margin_global"], meta["margin_by_stratum"])
    sf = slice_fraction(up, d["gpu_request"].values, LAD)
    w = d.copy(); w["gpu_request"] = (d["gpu_request"].values * sf).astype("float32")
    w["vram_entitled_gib"] = (d["vram_entitled_gib"].values * sf).astype("float32"); w["gpu_mem_per_gpu"] = w["vram_entitled_gib"] / w["gpu_request"]
    w["cpu_per_gpu"] = d["cpu_request_cores"] / w["gpu_request"]
    for col, cond in (("is_whole_gpu", w["gpu_request"] == 1), ("is_fractional", w["gpu_request"] < 1), ("is_multi_gpu", w["gpu_request"] > 1)): w[col] = cond.astype("float32")
    r0, r1 = cal.transform(B3T.predict(d)), cal.transform(B3T.predict(w)); dr = r1 - r0
    t = df[masks["train"]]; cnt = t.groupby([t["gpu_spec_public"].astype(str), t["gpu_request"].round(4)]).size()
    pos = pd.MultiIndex.from_arrays([w["gpu_spec_public"].astype(str), w["gpu_request"].round(4)]).isin(cnt[cnt >= 30].index)
    long = d["obs_hours"].values >= 24
    rows = []
    for pop, pm in (("all pods requesting >= 1 GPU", np.ones(len(d), bool)), ("pods observed >= 24 h (A's training population)", long)):
        res = d["reserved_gpu_hours"].values[pm]
        for rmax in (None, 0.10, 0.05, 0.02):
            ok = (sf[pm] < 1) & (np.ones(pm.sum(), bool) if rmax is None else dr[pm] <= rmax); sfp = np.where(ok, sf[pm], 1.0)
            rows.append(dict(population=pop, risk_max="no limit" if rmax is None else rmax, n=int(pm.sum()), share_shrunk=ok.mean(),
                             R_share=float(np.sum(res * (1 - sfp)) / np.sum(res)), mean_risk_before=r0[pm].mean(),
                             mean_risk_after=np.where(ok, r1[pm], r0[pm]).mean(), positivity_of_shrunk=pos[pm][ok].mean() if ok.any() else np.nan))
    POLICY = pd.DataFrame(rows).round(4); POLICY.to_csv(os.path.join(OUT_DIR, "AB_policy_whatif.csv"), index=False); display(POLICY)
    print(f"risk change when shrinking: median {np.median(dr):+.4f}, p90 {np.quantile(dr, .9):+.4f}; what-if rows with (GPU model, size) overlap: {pos.mean():.1%}")